In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Load dataset
voice_df = pd.read_csv('../../data/voice.csv')

print("Shape:", voice_df.shape)
display(voice_df.head())

print("\nDistribusi label:")
print(voice_df['label'].value_counts())

Shape: (3168, 21)


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male



Distribusi label:
label
male      1584
female    1584
Name: count, dtype: int64


In [2]:
# Pisahkan fitur dan target
X = voice_df.drop(columns=['label'])
y = voice_df['label']

# Konversi label menjadi numerik
encoder = LabelEncoder()
y = encoder.fit_transform(y)

print("Kelas:", encoder.classes_)
print("Jumlah fitur:", X.shape[1])

Kelas: ['female' 'male']
Jumlah fitur: 20


In [3]:
# Rasio split dan kernel yang diuji
split_ratios = {
    '70:30': 0.30,
    '80:20': 0.20
}

kernels = {
    'Linear': SVC(kernel='linear'),
    'Polynomial': SVC(kernel='poly', degree=3),
    'RBF': SVC(kernel='rbf')
}

results = []

for split_name, test_size in split_ratios.items():

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=42,
        stratify=y
    )

    for kernel_name, svm in kernels.items():

        model = make_pipeline(
            StandardScaler(),
            svm
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        results.append({
            'Split': split_name,
            'Kernel': kernel_name,
            'Accuracy': accuracy
        })

        print(
            f"Split {split_name} | "
            f"Kernel {kernel_name} | "
            f"Accuracy: {accuracy:.4f}"
        )

Split 70:30 | Kernel Linear | Accuracy: 0.9790
Split 70:30 | Kernel Polynomial | Accuracy: 0.9600
Split 70:30 | Kernel RBF | Accuracy: 0.9832
Split 80:20 | Kernel Linear | Accuracy: 0.9748
Split 80:20 | Kernel Polynomial | Accuracy: 0.9558
Split 80:20 | Kernel RBF | Accuracy: 0.9826


In [4]:
results_df = pd.DataFrame(results)

results_df['Accuracy (%)'] = (
    results_df['Accuracy'] * 100
).round(2)

results_df = results_df.sort_values(
    by=['Split', 'Accuracy'],
    ascending=[True, False]
)

display(
    results_df[
        ['Split', 'Kernel', 'Accuracy (%)']
    ].reset_index(drop=True)
)

,Split,Kernel,Accuracy (%)
0,70:30,RBF,98.32
1,70:30,Linear,97.90
2,70:30,Polynomial,96.00
3,80:20,RBF,98.26
4,80:20,Linear,97.48
5,80:20,Polynomial,95.58


In [5]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

In [6]:
train_dir = Path('../../data/training/')
test_dir = Path('../../data/test/')

def load_all_images(*directories):
    images = []
    labels = []

    for directory in directories:
        if not directory.exists():
            raise FileNotFoundError(
                f"Folder tidak ditemukan: {directory.resolve()}"
            )

        for class_dir in sorted(directory.iterdir()):
            if not class_dir.is_dir():
                continue

            label = class_dir.name.lower()

            for file in class_dir.iterdir():
                if file.suffix.lower() not in [
                    '.jpg', '.jpeg', '.png'
                ]:
                    continue

                image = cv2.imread(str(file))

                if image is not None:
                    images.append(image)
                    labels.append(label)

    return images, np.array(labels)

images, labels = load_all_images(train_dir, test_dir)

print("Jumlah gambar:", len(images))
print("Distribusi kelas:")
print(pd.Series(labels).value_counts())

Jumlah gambar: 400
Distribusi kelas:
day      200
night    200
Name: count, dtype: int64


In [7]:
def extract_histogram(image, bins=16):
    # OpenCV membaca gambar dalam format BGR
    hist_features = []

    for channel in range(3):
        hist = cv2.calcHist(
            [image],
            [channel],
            None,
            [bins],
            [0, 256]
        )

        # Normalisasi histogram
        hist = cv2.normalize(hist, hist).flatten()
        hist_features.extend(hist)

    return np.array(hist_features)

X = np.array([
    extract_histogram(image)
    for image in images
])

y = labels

print("Shape fitur:", X.shape)
print("Shape label:", y.shape)

Shape fitur: (400, 48)
Shape label: (400,)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Jumlah data training:", len(X_train))
print("Jumlah data testing:", len(X_test))

Jumlah data training: 320
Jumlah data testing: 80


In [9]:
baseline_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf'))
])

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_pred)

print(f"Baseline accuracy: {baseline_accuracy:.4f}")
print(f"Baseline accuracy: {baseline_accuracy * 100:.2f}%")

Baseline accuracy: 1.0000
Baseline accuracy: 100.00%


In [10]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf'))
])

param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 0.001, 0.01, 0.1, 1]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best CV accuracy: {grid_search.best_score_ * 100:.2f}%")

Best parameters: {'svm__C': 10, 'svm__gamma': 'scale'}
Best CV accuracy: 99.69%


In [11]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test, y_pred)

print(f"Test accuracy: {tuned_accuracy * 100:.2f}%")
print("\nClassification report:")
print(classification_report(y_test, y_pred))

Test accuracy: 98.75%

Classification report:
              precision    recall  f1-score   support

         day       1.00      0.97      0.99        40
       night       0.98      1.00      0.99        40

    accuracy                           0.99        80
   macro avg       0.99      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80



In [12]:
comparison_df = pd.DataFrame({
    'Model': ['SVM RBF Baseline', 'SVM RBF Tuned'],
    'Accuracy (%)': [
        baseline_accuracy * 100,
        tuned_accuracy * 100
    ]
})

comparison_df['Accuracy (%)'] = comparison_df['Accuracy (%)'].round(2)

display(comparison_df)

,Model,Accuracy (%)
0,SVM RBF Baseline,100.00
1,SVM RBF Tuned,98.75


### Kesimpulan

**1. Klasifikasi voice.csv**

Pengujian dilakukan menggunakan algoritma Support Vector Machine (SVM) dengan kernel Linear, Polynomial, dan RBF pada rasio pembagian data 70:30 dan 80:20. Performa setiap model dievaluasi menggunakan metrik akurasi.

Berdasarkan hasil pengujian, model dengan akurasi tertinggi adalah kernel **(nama kernel)** pada rasio split **(rasio split)** dengan akurasi sebesar **(nilai akurasi)%**.

**2. Klasifikasi siang dan malam**

Klasifikasi gambar siang dan malam dilakukan menggunakan SVM dengan kernel RBF dan fitur histogram warna. Dataset dibagi dengan rasio 80:20. Selanjutnya, dilakukan hyperparameter tuning terhadap nilai `C` dan `gamma`.

Model baseline memperoleh akurasi sebesar **(akurasi baseline)%**, sedangkan model setelah tuning memperoleh akurasi sebesar **(akurasi tuning)%** pada data testing. Hyperparameter terbaik yang diperoleh adalah **(parameter terbaik)**.

Hasil tersebut menunjukkan bahwa penggunaan fitur histogram dan pemilihan hyperparameter dapat memengaruhi performa klasifikasi. Model terbaik ditentukan berdasarkan akurasi pada data testing, bukan hanya berdasarkan hasil cross-validation.
